# Exercises XP Ninja : W9_D1

## XP Ninja: Build an Intelligent Document Assistant

### What You’ll Learn
- Combine foundational and advanced prompt engineering techniques  
- Apply Chain-of-Thought, role prompting, and memory chaining in a real-world scenario  
- Design a dynamic, multi-step LLM workflow that adapts to user input  
- Mitigate model limitations like hallucination and bias  

### What You Will Create
An LLM-powered **Document Assistant** that:
- Takes a long input document (below)  
- Extracts key sections and summarizes them using appropriate prompt patterns  
- Allows users to ask follow-up questions based on the document  
- Maintains context across turns and **reflects on or critiques** its own outputs  

---

### Instructions — Use This Document as Input
**Document Text:**

> **Service Agreement – Excerpt**  
> This Service Agreement (“Agreement”) is made effective as of **March 1, 2025**, by and between **BrightLine Technologies Ltd.** (“Provider”) and **NovaWare Systems Inc.** (“Client”).  
>   
> **Scope of Work:** Provider shall deliver **cloud infrastructure management services**, including **monitoring, incident response, and monthly reporting**, as described in Exhibit A.  
> **Payment Terms:** Client agrees to pay a **fixed monthly fee of \$12,000**, payable within **30 days** of receipt of invoice. **Late payments** will incur a **2% penalty per month**.  
> **Term and Termination:** This Agreement shall commence on **March 1, 2025**, and remain in effect for **12 months**. Either party may **terminate with 30 days’ written notice**.  
> **Confidentiality:** Both parties agree to **protect the confidentiality** of proprietary or sensitive information shared during the course of the engagement.  
> **Limitation of Liability:** Provider’s total liability shall **not exceed the fees paid by Client in the 3 months prior to a claim**. Provider is **not liable** for **indirect or consequential damages**.  
> **Governing Law:** This Agreement shall be **governed by the laws of the State of California**.

---

### Step 1: Initial Summary Prompt
**Task:** Write a **few-shot** or **Chain-of-Thought** prompt to summarize the document in plain English.

Your summary must include:
- **Key responsibilities** of each party  
- **Payment terms**  
- **Termination clauses**  
- **Liability limitations**  

---

### Step 2: Role-Based Follow-Up Q&A
**Task:** Write a **role-based prompt** where the assistant acts as a **contract lawyer** answering user questions about this agreement.  
*Example question:* “Can you explain the limitation of liability clause?”  
Use **Narrative-of-Thought** or **Instance-Adaptive CoT** to deliver engaging and helpful responses.

---

### Step 3: Memory Integration
**Task:** Simulate **context chaining** so the assistant remembers the summary and answers questions accordingly.

Choose and apply **one method**:
- **Prior message passing**  
- **Structured conversation history**  
- *(Or describe how you would use **vector store retrieval**)*

---

### Step 4: Mitigation & Refinement
**Task:** Add a **self-reflection prompt** where the assistant **reviews and critiques its own answer** for accuracy and clarity.  
**Optional:** Include a **multi-agent critique** step to catch potential legal misinterpretations.

## Step 1: Initial Summary Prompt

### Title: Initial Summary Prompt (Few-Shot CoT, English)
### Goal: Summarize the contract in plain English with a fixed schema.
### Safety: Use ONLY the provided document. If info is missing, say "Not specified".

[ROLE] Act as a contract analyst.

[DOCUMENT]
Service Agreement – Excerpt
Effective date: March 1, 2025
Parties: BrightLine Technologies Ltd. ("Provider") and NovaWare Systems Inc. ("Client").

Scope of Work: Provider shall deliver cloud infrastructure management services, including monitoring, incident response, and monthly reporting, as described in Exhibit A.
Payment Terms: Client agrees to pay a fixed monthly fee of $12,000, payable within 30 days of receipt of invoice. Late payments will incur a 2% penalty per month.
Term and Termination: This Agreement shall commence on March 1, 2025, and remain in effect for 12 months. Either party may terminate with 30 days’ written notice.
Confidentiality: Both parties agree to protect the confidentiality of proprietary or sensitive information shared during the course of the engagement.
Limitation of Liability: Provider’s total liability shall not exceed the fees paid by Client in the 3 months prior to a claim. Provider is not liable for indirect or consequential damages.
Governing Law: This Agreement shall be governed by the laws of the State of California.

[FEW-SHOT DEMO — FORMAT ONLY]
Input (mini-sample): "Payment due within 15 days; termination with 10 days’ notice."
Output (schema):
- Responsibilities (Provider): ...
- Responsibilities (Client): ...
- Payment Terms: ...
- Term & Termination: ...
- Liability Limitations: ...
- Confidentiality: ...
- Governing Law: ...

[TASK]
Produce a concise summary in this exact schema:
- Responsibilities (Provider): <bullet list, ≤3 bullets>
- Responsibilities (Client): <bullet list, ≤3 bullets>
- Payment Terms: <1–2 sentences>
- Term & Termination: <1–2 sentences>
- Liability Limitations: <1 sentence>
- Confidentiality: <1 sentence>
- Governing Law: <1 sentence>

[REASONING STYLE]
Provide a short "Reasoning Summary" of ≤3 bullets (no hidden steps).

[CONSTRAINTS]
- Plain English, no legalese.
- Quote numbers/dates exactly as in the document.
- If any field is absent, write "Not specified".

[OUTPUT]
Return ONLY the schema and the short "Reasoning Summary".

### Interpretation:
A fixed schema + brief reasoning summary keeps the model grounded, while explicit constraints (numbers/dates, “Not specified”) prevent drift.

## Step 2: Role-Based Follow-Up Q&A

### Title: Role-Based Contract Q&A (Narrative-of-Thought, English)
### Goal: Answer user questions about this agreement as a contract lawyer; be clear and document-bound.

[ROLE]
You are a contract lawyer explaining clauses to a non-lawyer audience.

[CONTEXT — DOCUMENT (same as Step 1)]
<<<PASTE THE CONTRACT EXCERPT HERE OR REFERENCE A DOC VARIABLE>>>

[USER QUESTION]
Q: "<<USER_QUESTION>>"  (e.g., "Can you explain the limitation of liability clause?")

[APPROACH: Narrative-of-Thought — concise]
- Identify the relevant clause(s) by name.
- Paraphrase the clause in plain English.
- Highlight exact numbers/dates/percentages if present.
- State a practical implication for each party.
- If the question asks for something not in the document, say "Not specified in the excerpt."

[ANSWER FORMAT]
- Clause referenced: <name>
- Plain-English explanation: <2–3 sentences>
- Practical implications (Provider/Client): <2 bullets>
- Caveat: <1 sentence if limits/ambiguities apply>

[GUARDRAILS]
- Use ONLY the provided text; do not infer beyond it.
- If the user asks for legal advice, add: "This is an informational summary, not legal advice."

### Interpretation:
Role + audience + fixed output blocks yields consistent, user-friendly answers and avoids overreach beyond the excerpt.

## Step 3: Memory Integration

Chosen method: Structured conversation history (key slots). It is deterministic, auditable, and easy to update.

```{
  "memory": {
    "doc_id": "service_agreement_march_2025",
    "summary": {
      "provider_responsibilities": [
        "Cloud infrastructure monitoring",
        "Incident response",
        "Monthly reporting (per Exhibit A)"
      ],
      "client_responsibilities": [
        "Pay $12,000 monthly within 30 days of invoice"
      ],
      "payment_terms": "Fixed $12,000/month; 30-day payment window; 2% monthly late penalty.",
      "term_termination": "Starts Mar 1, 2025; 12 months; either party may terminate with 30 days' written notice.",
      "liability": "Capped at fees paid in the prior 3 months; excludes indirect/consequential damages.",
      "confidentiality": "Both parties must protect proprietary/sensitive information.",
      "governing_law": "State of California."
    },
    "qa_history": [
      { "q": "What triggers late fees?", "a": "Payment after 30 days incurs 2% per month." }
    ]
  }
}```


### Title: Next-Turn Prompt with Structured Memory
### Goal: Use stored summary + QA history to personalize and stay consistent.

[ROLE] Contract assistant with memory.

[MEMORY SNAPSHOT]
<<<STRUCTURED_MEMORY_JSON_FROM_ABOVE>>>

[NEW USER INPUT]
"<<USER_QUESTION>>"

[INSTRUCTIONS]
- Check memory.summary for relevant clause(s) first.
- If needed, cite exact figures/dates from memory.summary or the base document.
- If a detail is missing from memory, search the base document text attached to the session (if available). If not available, say "Not specified in the excerpt."
- Append a one-line "Memory Update Proposal" if this Q&A adds a stable fact.

[OUTPUT]
- Answer: <concise, plain English>
- Reference: <which clause/section was used>
- Memory Update Proposal: <one line or "none">

### Interpretation:
By slotting key facts into memory, you ensure continuity across turns and reduce re-reading. The “Update Proposal” enforces careful growth of memory.

## Step 4: Mitigation & Refinement

### Title: Self-Reflection Checklist
### Goal: Critique the assistant’s last answer for accuracy/clarity BEFORE sending.

[INPUTS]
- Last answer draft: <<<ANSWER_DRAFT>>>
- Document excerpt: <<<DOCUMENT_TEXT_OR_ID>>>

[CHECKLIST]
1) Document-bounded? Any claim not directly supported? If yes, revise to remove it.
2) Numbers/dates exact? (e.g., $12,000; 30 days; 2%/month; 12 months; March 1, 2025)
3) Plain English and role-appropriate tone?
4) Ambiguities identified with “Not specified” where needed?
5) Output format followed (blocks, bullets, word limits)?

[ACTION]
- If any item fails, produce a corrected answer. Otherwise, return “No changes needed.”

### Title: Optional Multi-Agent Critique
### Pattern: Analyst → Critic → Resolver

[ANALYST]
Produce the best possible answer to the user’s question (document-bound, plain English, exact figures).

[CRITIC]
Review the Analyst’s answer strictly against the document. Flag:
- Unsupported claims
- Missing key numbers/dates
- Ambiguous statements without “Not specified”
Suggest concrete edits (one-line each).

[RESOLVER]
Incorporate Critic’s edits. Return the final answer only, then list 1 sentence: "What changed and why."

### Interpretation:
A lightweight checklist catches common mistakes; a short Analyst→Critic→Resolver loop improves reliability without heavy overhead.

### Quick Run Example (Filled Output Snippet)

### Title: Example Answer to "Explain the limitation of liability clause"
Clause referenced: Limitation of Liability
Plain-English explanation: The Provider’s total liability is capped at the amount the Client paid in the 3 months before a claim. The Provider is not responsible for indirect or consequential damages.
Practical implications:
- Provider: Exposure limited to a 3-month fee cap.
- Client: Cannot recover indirect/consequential damages even if losses are larger.
Caveat: This summary is based on the excerpt and is informational, not legal advice.

### Interpretation:
This example mirrors the required output blocks, quotes numbers exactly, and adds a small caveat to avoid over-interpretation.